# Phase 4 and 5 - Model Access and Agents

Project: ComplaintIQ

Work through this notebook incrementally. Validate each step before proceeding.

In [1]:
from pathlib import Path
PROJECT_ROOT = Path.home() / 'complaintiq'
print('Project root:', PROJECT_ROOT)
print('Exists:', PROJECT_ROOT.exists())

Project root: /home/sagemaker-user/complaintiq
Exists: True


In [21]:
from pathlib import Path
import os
import sys
import json
import time
import hashlib
import sqlite3
import threading
import statistics
from datetime import datetime, timezone
from src.config import (
    ROOT,
    RAW,
    CFPB_RAW,
    ECFR_RAW,
    PDF_RAW,
    PROC,
    UA,
    INDEX
)
ROOT = Path("/home/sagemaker-user/complaintiq")
os.chdir(ROOT)

SRC = ROOT / "src"
STATE = ROOT / "state"
REPORTS = ROOT / "reports"

for path in [SRC, STATE, REPORTS]:
    path.mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project root:", ROOT)
print("Source directory:", SRC)
print("State directory:", STATE)
print("Reports directory:", REPORTS)

Project root: /home/sagemaker-user/complaintiq
Source directory: /home/sagemaker-user/complaintiq/src
State directory: /home/sagemaker-user/complaintiq/state
Reports directory: /home/sagemaker-user/complaintiq/reports


In [4]:
from config import REGION, STATE as CONFIG_STATE

MODELS_PATH = ROOT / "models.json"

assert REGION == "us-east-1", f"Unexpected region: {REGION}"
assert MODELS_PATH.exists(), "models.json is missing"

MODELS = json.loads(MODELS_PATH.read_text())

print("Region:", REGION)
print("Resolved model roles:")

for role, model_id in MODELS.items():
    print(f"{role:10s}: {model_id}")

assert MODELS.get("worker"), "Worker model is not configured"
assert MODELS.get("judge"), "Judge model is not configured"

print("\nPASS — configuration and model IDs loaded")

Region: us-east-1
Resolved model roles:
worker    : us.anthropic.claude-sonnet-4-6
judge     : us.anthropic.claude-opus-4-8
fast      : us.anthropic.claude-sonnet-4-6
embed     : amazon.titan-embed-text-v2:0

PASS — configuration and model IDs loaded


In [9]:
import boto3

session = boto3.Session(region_name="us-east-1")
identity = session.client("sts").get_caller_identity()

print("AWS Account:", identity["Account"])
print("AWS ARN:", identity["Arn"])
print("AWS Region:", session.region_name)

print("\nBedrock client region:", llm.brt.meta.region_name)

assert llm.brt.meta.region_name == "us-east-1"

print("\nPASS — Bedrock client is configured for us-east-1")

AWS Account: 456427397380
AWS ARN: arn:aws:sts::456427397380:assumed-role/AmazonSageMaker-ExecutionRole-20261106T002068/SageMaker
AWS Region: us-east-1

Bedrock client region: us-east-1

PASS — Bedrock client is configured for us-east-1


In [6]:
import boto3
import botocore
import pydantic

from botocore.config import Config
from botocore.exceptions import ClientError
from pydantic import BaseModel, Field, ValidationError

print("boto3:", boto3.__version__)
print("botocore:", botocore.__version__)
print("pydantic:", pydantic.__version__)

print("PASS — required libraries imported")

boto3: 1.43.106
botocore: 1.43.106
pydantic: 2.14.1
PASS — required libraries imported


In [7]:
%%writefile src/llm.py

import json
import hashlib
import sqlite3
import time
import threading
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from pydantic import BaseModel, ValidationError

from config import REGION, STATE


# ============================================================
# 1. BEDROCK CLIENT
# ============================================================

brt = boto3.client(
    "bedrock-runtime",
    region_name=REGION,
    config=Config(
        retries={
            "max_attempts": 12,
            "mode": "adaptive"
        },
        read_timeout=180,
        connect_timeout=15
    )
)


# ============================================================
# 2. MODEL PRICING
# ============================================================
# Prices are USD per 1,000 tokens.
#
# Keep these as None until you enter the applicable prices
# for your actual model/inference profile and pricing tier.
#
# Do not use guessed prices for the final cost report.

PRICE_PER_1K = {
    "sonnet-4-6": (None, None),
    "opus-4-8": (None, None),
    "haiku": (None, None)
}


def configure_price(model_family, input_price, output_price):
    """
    Configure USD price per 1,000 input/output tokens.
    """
    if input_price < 0 or output_price < 0:
        raise ValueError("Prices cannot be negative")

    PRICE_PER_1K[model_family] = (
        float(input_price),
        float(output_price)
    )


def _price_for_model(model_id):
    model_id = model_id.lower()

    for family, prices in PRICE_PER_1K.items():
        if family in model_id:
            return prices

    return None, None


# ============================================================
# 3. USAGE AND COST LEDGER
# ============================================================

class Ledger:

    def __init__(self):
        self.rows = []
        self.lock = threading.Lock()

    def add(self, model, tin, tout, ms):
        with self.lock:
            self.rows.append({
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "model": model,
                "input_tokens": int(tin),
                "output_tokens": int(tout),
                "latency_ms": int(ms)
            })

    def total_tokens(self):
        with self.lock:
            return sum(r["input_tokens"] + r["output_tokens"]
                       for r in self.rows)

    def total_input_tokens(self):
        with self.lock:
            return sum(r["input_tokens"] for r in self.rows)

    def total_output_tokens(self):
        with self.lock:
            return sum(r["output_tokens"] for r in self.rows)

    def usd(self):
        total = 0.0
        unknown_models = set()

        with self.lock:
            rows = list(self.rows)

        for row in rows:
            pi, po = _price_for_model(row["model"])

            if pi is None or po is None:
                unknown_models.add(row["model"])
                continue

            total += (
                row["input_tokens"] / 1000 * pi
                + row["output_tokens"] / 1000 * po
            )

        return round(total, 6)

    def latency_summary(self):
        with self.lock:
            values = [r["latency_ms"] for r in self.rows]

        if not values:
            return {
                "calls": 0,
                "average_ms": 0,
                "p50_ms": 0,
                "p95_ms": 0
            }

        ordered = sorted(values)

        return {
            "calls": len(values),
            "average_ms": round(statistics.mean(values), 2),
            "p50_ms": round(
                ordered[int((len(ordered) - 1) * 0.50)], 2
            ),
            "p95_ms": round(
                ordered[int((len(ordered) - 1) * 0.95)], 2
            )
        }

    def summary(self):
        with self.lock:
            rows = list(self.rows)

        return {
            "calls": len(rows),
            "input_tokens": sum(r["input_tokens"] for r in rows),
            "output_tokens": sum(r["output_tokens"] for r in rows),
            "total_tokens": sum(
                r["input_tokens"] + r["output_tokens"]
                for r in rows
            ),
            "estimated_cost_usd": self.usd(),
            "latency": self.latency_summary(),
            "models_used": sorted(set(r["model"] for r in rows))
        }

    def to_list(self):
        with self.lock:
            return list(self.rows)


LEDGER = Ledger()


# ============================================================
# 4. SQLITE RESPONSE CACHE
# ============================================================

CACHE_PATH = Path(STATE) / "llm_cache.sqlite"
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

_cache_lock = threading.Lock()

_cache = sqlite3.connect(
    str(CACHE_PATH),
    check_same_thread=False
)

_cache.execute("""
    CREATE TABLE IF NOT EXISTS c (
        k TEXT PRIMARY KEY,
        v TEXT NOT NULL,
        created_at TEXT
    )
""")

_cache.commit()


def _cache_get(key):

    with _cache_lock:
        row = _cache.execute(
            "SELECT v FROM c WHERE k = ?",
            (key,)
        ).fetchone()

    return row[0] if row else None


def _cache_set(key, value):

    with _cache_lock:
        _cache.execute(
            """
            INSERT OR REPLACE INTO c (k, v, created_at)
            VALUES (?, ?, ?)
            """,
            (
                key,
                value,
                datetime.now(timezone.utc).isoformat()
            )
        )

        _cache.commit()


def cache_stats():

    with _cache_lock:
        count = _cache.execute(
            "SELECT COUNT(*) FROM c"
        ).fetchone()[0]

    return {
        "cache_path": str(CACHE_PATH),
        "cached_responses": count
    }


# ============================================================
# 5. TEMPERATURE COMPATIBILITY
# ============================================================

_no_temperature = set()


# ============================================================
# 6. BEDROCK CHAT
# ============================================================

def chat(
    model_id,
    system,
    user,
    max_tokens=1500,
    temperature=0.0,
    use_cache=True
):
    """
    Central Bedrock Converse wrapper.

    Features:
      - Explicit region
      - Adaptive SDK retries
      - SQLite response cache
      - Token accounting
      - Latency accounting
      - Temperature compatibility retry
    """

    cache_payload = json.dumps(
        [
            model_id,
            system,
            user,
            max_tokens,
            temperature
        ],
        sort_keys=True
    )

    key = hashlib.sha256(
        cache_payload.encode("utf-8")
    ).hexdigest()

    # --------------------------------------------------------
    # CACHE LOOKUP
    # --------------------------------------------------------

    if use_cache:
        cached = _cache_get(key)

        if cached is not None:
            return cached

    # --------------------------------------------------------
    # REQUEST CONFIGURATION
    # --------------------------------------------------------

    inference_config = {
        "maxTokens": max_tokens
    }

    if model_id not in _no_temperature:
        inference_config["temperature"] = temperature

    # --------------------------------------------------------
    # BEDROCK CALL
    # --------------------------------------------------------

    start = time.perf_counter()

    try:

        response = brt.converse(
            modelId=model_id,
            system=[
                {"text": system}
            ],
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"text": user}
                    ]
                }
            ],
            inferenceConfig=inference_config
        )

    except ClientError as exc:

        message = str(exc).lower()

        # Some models reject the temperature parameter.
        # Remember this for future calls and retry once
        # without temperature.

        if (
            "temperature" in message
            and model_id not in _no_temperature
        ):

            _no_temperature.add(model_id)

            return chat(
                model_id=model_id,
                system=system,
                user=user,
                max_tokens=max_tokens,
                temperature=temperature,
                use_cache=use_cache
            )

        raise

    latency_ms = int(
        (time.perf_counter() - start) * 1000
    )

    # --------------------------------------------------------
    # RESPONSE EXTRACTION
    # --------------------------------------------------------

    response_text = response["output"]["message"]["content"][0]["text"]

    usage = response.get("usage", {})

    input_tokens = usage.get("inputTokens", 0)
    output_tokens = usage.get("outputTokens", 0)

    # --------------------------------------------------------
    # LEDGER UPDATE
    # --------------------------------------------------------

    LEDGER.add(
        model=model_id,
        tin=input_tokens,
        tout=output_tokens,
        ms=latency_ms
    )

    # --------------------------------------------------------
    # CACHE SAVE
    # --------------------------------------------------------

    if use_cache:
        _cache_set(key, response_text)

    return response_text


# ============================================================
# 7. STRUCTURED JSON OUTPUT WITH PYDANTIC VALIDATION
# ============================================================

def chat_json(
    model_id,
    system,
    user,
    schema: type[BaseModel],
    **kwargs
):
    """
    Requests JSON from the model and validates it using
    Pydantic.

    The model gets one repair attempt if its first response
    is invalid.
    """

    json_system = (
        system
        + "\n\nReturn ONLY one valid JSON object. "
          "Do not include markdown fences or prose."
    )

    raw = chat(
        model_id=model_id,
        system=json_system,
        user=user,
        **kwargs
    )

    for attempt in range(2):

        try:

            start = raw.find("{")
            end = raw.rfind("}")

            if start == -1 or end == -1 or end < start:
                raise ValueError("No JSON object found")

            blob = raw[start:end + 1]

            return schema.model_validate_json(blob)

        except (ValueError, ValidationError) as exc:

            if attempt == 1:
                raise

            repair_user = (
                user
                + "\n\nYour previous response was invalid."
                + f"\nValidation error: {str(exc)}"
                + "\nReturn valid JSON matching this schema:"
                + json.dumps(schema.model_json_schema(), indent=2)
            )

            raw = chat(
                model_id=model_id,
                system=json_system,
                user=repair_user,
                use_cache=False,
                **kwargs
            )


# ============================================================
# 8. REPORTING HELPERS
# ============================================================

def get_usage_report():

    return {
        "ledger": LEDGER.summary(),
        "cache": cache_stats(),
        "models_without_temperature": sorted(_no_temperature)
    }


def reset_ledger():
    """
    Clears in-memory metrics only.
    Does NOT delete the SQLite response cache.
    """
    with LEDGER.lock:
        LEDGER.rows.clear()

Overwriting src/llm.py


In [25]:
import importlib
import llm

importlib.reload(llm)

from llm import (
    chat,
    chat_json,
    statistics,
    LEDGER,
    PRICE_PER_1K,
    configure_price,
    cache_stats,
    get_usage_report
)

print("Bedrock wrapper loaded")
print("Region:", llm.REGION)
print("Cache:", llm.CACHE_PATH)
print("Pricing configuration:", PRICE_PER_1K)

assert llm.REGION == "us-east-1"
assert llm.CACHE_PATH.exists()

print("\nPASS — Phase 4 wrapper is importable")

Bedrock wrapper loaded
Region: us-east-1
Cache: /home/sagemaker-user/complaintiq/state/llm_cache.sqlite
Pricing configuration: {'sonnet-4-6': (None, None), 'opus-4-8': (None, None), 'haiku': (None, None)}

PASS — Phase 4 wrapper is importable


In [10]:
from pydantic import BaseModel, Field

class LLMTestResult(BaseModel):
    status: str
    category: str
    confidence: float = Field(ge=0, le=1)
    explanation: str


print(LLMTestResult.model_json_schema())

{'properties': {'status': {'title': 'Status', 'type': 'string'}, 'category': {'title': 'Category', 'type': 'string'}, 'confidence': {'maximum': 1, 'minimum': 0, 'title': 'Confidence', 'type': 'number'}, 'explanation': {'title': 'Explanation', 'type': 'string'}}, 'required': ['status', 'category', 'confidence', 'explanation'], 'title': 'LLMTestResult', 'type': 'object'}


In [12]:
WORKER_MODEL = MODELS["worker"]

worker_system = """
You are a structured-output testing assistant.

Classify the supplied test message.
Return a JSON object with:
- status
- category
- confidence
- explanation

Confidence must be between 0 and 1.
"""

worker_user = """
Test message:
A customer reports that an electronic funds transfer
was not credited to their checking account.

Return the classification as JSON.
"""

worker_result = chat_json(
    model_id=WORKER_MODEL,
    system=worker_system,
    user=worker_user,
    schema=LLMTestResult,
    max_tokens=300,
    temperature=0.0
)

print(worker_result)
print("\nValidated output:")
print(worker_result.model_dump())

assert isinstance(worker_result, LLMTestResult)
assert 0 <= worker_result.confidence <= 1

print("\nPASS — worker model returned Pydantic-valid output")

status='issue' category='banking_transaction_dispute' confidence=0.97 explanation='The message describes a customer reporting a missing electronic funds transfer (EFT) that was not credited to their checking account. This is a clear banking/financial transaction issue requiring investigation into a failed or missing credit posting.'

Validated output:
{'status': 'issue', 'category': 'banking_transaction_dispute', 'confidence': 0.97, 'explanation': 'The message describes a customer reporting a missing electronic funds transfer (EFT) that was not credited to their checking account. This is a clear banking/financial transaction issue requiring investigation into a failed or missing credit posting.'}

PASS — worker model returned Pydantic-valid output


In [13]:
JUDGE_MODEL = MODELS["judge"]

judge_system = """
You are a response-quality evaluator.

Evaluate the supplied answer.
Return JSON with:
- status
- category
- confidence
- explanation

Return only a JSON object.
"""

judge_user = """
Question:
What should a bank investigate when a customer reports
an electronic funds transfer error?

Candidate answer:
The bank should investigate the reported transfer error
and determine whether the applicable error-resolution
requirements have been met.

Evaluate whether the answer is relevant and appropriate.
"""

judge_result = chat_json(
    model_id=JUDGE_MODEL,
    system=judge_system,
    user=judge_user,
    schema=LLMTestResult,
    max_tokens=300,
    temperature=0.0
)

print(judge_result.model_dump())

assert isinstance(judge_result, LLMTestResult)

print("\nPASS — judge model returned Pydantic-valid output")
print("Models remembered as not supporting temperature:",
      llm._no_temperature)

{'status': 'fail', 'category': 'insufficient_detail', 'confidence': 0.85, 'explanation': "The answer is relevant to the question but is circular and lacks substantive content. It essentially restates the question ('investigate the reported transfer error') without specifying what the bank should actually examine. A proper answer would identify concrete investigation elements, such as verifying whether an error occurred, the type and amount of the transfer, the date and location, the parties involved, provisional crediting obligations, applicable timeframes (e.g., under Regulation E), notification requirements, and documentation review. As written, it provides no actionable or informative detail."}

PASS — judge model returned Pydantic-valid output
Models remembered as not supporting temperature: {'us.anthropic.claude-opus-4-8'}


In [14]:
cache_test_system = """
You are a test assistant.
Return one short sentence.
"""

cache_test_user = """
Explain in one sentence why a bank should investigate
a reported electronic funds transfer error.
"""

before_calls = len(LEDGER.rows)

first_response = chat(
    model_id=WORKER_MODEL,
    system=cache_test_system,
    user=cache_test_user,
    max_tokens=100,
    temperature=0.0,
    use_cache=True
)

after_first = len(LEDGER.rows)

second_response = chat(
    model_id=WORKER_MODEL,
    system=cache_test_system,
    user=cache_test_user,
    max_tokens=100,
    temperature=0.0,
    use_cache=True
)

after_second = len(LEDGER.rows)

print("First response:")
print(first_response)

print("\nSecond response:")
print(second_response)

print("\nLedger before:", before_calls)
print("Ledger after first call:", after_first)
print("Ledger after second call:", after_second)

assert first_response == second_response
assert after_first == before_calls + 1
assert after_second == after_first

print("\nPASS — second request was served from cache")

First response:
A bank should investigate a reported electronic funds transfer error to verify its validity, correct any mistakes, and protect the customer's funds as required by law.

Second response:
A bank should investigate a reported electronic funds transfer error to verify its validity, correct any mistakes, and protect the customer's funds as required by law.

Ledger before: 2
Ledger after first call: 3
Ledger after second call: 3

PASS — second request was served from cache


In [17]:
import uuid

cache_test_id = str(uuid.uuid4())

cache_test_system = """
You are a test assistant.
Return one short sentence.
"""

cache_test_user = f"""
Cache validation test ID: {cache_test_id}

Explain in one sentence why a bank should investigate
a reported electronic funds transfer error.
"""

before_calls = len(LEDGER.rows)

# First request — Expected CACHE MISS
first_response = chat(
    model_id=WORKER_MODEL,
    system=cache_test_system,
    user=cache_test_user,
    max_tokens=100,
    temperature=0.0,
    use_cache=True
)

after_first = len(LEDGER.rows)

# Second request — Expected CACHE HIT
second_response = chat(
    model_id=WORKER_MODEL,
    system=cache_test_system,
    user=cache_test_user,
    max_tokens=100,
    temperature=0.0,
    use_cache=True
)

after_second = len(LEDGER.rows)

print("First response:")
print(first_response)

print("\nSecond response:")
print(second_response)

print("\nLedger before:", before_calls)
print("Ledger after first call:", after_first)
print("Ledger after second call:", after_second)

# Validation
assert first_response == second_response, "Responses do not match"
assert after_first == before_calls + 1, "First request did not create exactly one ledger entry"
assert after_second == after_first, "Second request unexpectedly created another ledger entry"

print("\nPASS — First request was a cache miss.")
print("PASS — Second request was served from cache.")
print("PASS — Only one Bedrock invocation was recorded.")

First response:
A bank should investigate a reported electronic funds transfer error to verify its validity and, if confirmed, correct the error and restore the customer's account to its proper balance as required by law.

Second response:
A bank should investigate a reported electronic funds transfer error to verify its validity and, if confirmed, correct the error and restore the customer's account to its proper balance as required by law.

Ledger before: 3
Ledger after first call: 4
Ledger after second call: 4

PASS — First request was a cache miss.
PASS — Second request was served from cache.
PASS — Only one Bedrock invocation was recorded.


In [18]:
print(cache_stats())

import sqlite3

cache_con = sqlite3.connect(str(llm.CACHE_PATH))

cache_rows = cache_con.execute("""
    SELECT COUNT(*) FROM c
""").fetchone()[0]

print("Cached responses:", cache_rows)

cache_con.close()

assert cache_rows >= 1

print("\nPASS — SQLite cache contains saved responses")

{'cache_path': '/home/sagemaker-user/complaintiq/state/llm_cache.sqlite', 'cached_responses': 4}
Cached responses: 4

PASS — SQLite cache contains saved responses


In [26]:
usage_report = get_usage_report()

print(json.dumps(
    usage_report,
    indent=2,
    default=str
))

{
  "ledger": {
    "calls": 0,
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0,
    "estimated_cost_usd": 0.0,
    "latency": {
      "calls": 0,
      "average_ms": 0,
      "p50_ms": 0,
      "p95_ms": 0
    },
    "models_used": []
  },
  "cache": {
    "cache_path": "/home/sagemaker-user/complaintiq/state/llm_cache.sqlite",
    "cached_responses": 4
  },
  "models_without_temperature": []
}


In [27]:
for role in ["worker", "judge"]:
    print(role, "=", MODELS[role])

print("\nCurrent pricing:")
print(PRICE_PER_1K)

worker = us.anthropic.claude-sonnet-4-6
judge = us.anthropic.claude-opus-4-8

Current pricing:
{'sonnet-4-6': (None, None), 'opus-4-8': (None, None), 'haiku': (None, None)}


In [28]:
# Replace None with verified USD prices per 1,000 tokens.
#
# Example format only:
# configure_price("sonnet-4-6", INPUT_PRICE, OUTPUT_PRICE)
#
# Do not run with invented values.

# configure_price("sonnet-4-6", VERIFIED_INPUT_PRICE, VERIFIED_OUTPUT_PRICE)
# configure_price("opus-4-8", VERIFIED_INPUT_PRICE, VERIFIED_OUTPUT_PRICE)
# configure_price("haiku", VERIFIED_INPUT_PRICE, VERIFIED_OUTPUT_PRICE)

print("Pricing configuration:")
print(PRICE_PER_1K)

Pricing configuration:
{'sonnet-4-6': (None, None), 'opus-4-8': (None, None), 'haiku': (None, None)}


In [29]:
from datetime import datetime, timezone

final_report = {
    "phase": 4,
    "phase_name": "Bedrock Access Layer",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "region": REGION,
    "models": {
        "worker": WORKER_MODEL,
        "judge": JUDGE_MODEL
    },
    "tests": {
        "configuration_loaded": True,
        "bedrock_region_verified": llm.brt.meta.region_name == "us-east-1",
        "worker_pydantic_validation": isinstance(
            worker_result, LLMTestResult
        ),
        "judge_pydantic_validation": isinstance(
            judge_result, LLMTestResult
        ),
        "cache_hit_verified": (
            first_response == second_response
            and after_second == after_first
        ),
        "sqlite_cache_exists": llm.CACHE_PATH.exists()
    },
    "usage": get_usage_report(),
    "pricing_configuration": PRICE_PER_1K
}

report_path = REPORTS / "phase4_llm_tests.json"

report_path.write_text(
    json.dumps(final_report, indent=2, default=str)
)

print("Report saved:", report_path)
print(json.dumps(final_report, indent=2, default=str))

Report saved: /home/sagemaker-user/complaintiq/reports/phase4_llm_tests.json
{
  "phase": 4,
  "phase_name": "Bedrock Access Layer",
  "timestamp_utc": "2026-11-06T07:46:48.852994+00:00",
  "region": "us-east-1",
  "models": {
    "worker": "us.anthropic.claude-sonnet-4-6",
    "judge": "us.anthropic.claude-opus-4-8"
  },
  "tests": {
    "configuration_loaded": true,
    "bedrock_region_verified": true,
    "worker_pydantic_validation": true,
    "judge_pydantic_validation": true,
    "cache_hit_verified": true,
    "sqlite_cache_exists": true
  },
  "usage": {
    "ledger": {
      "calls": 0,
      "input_tokens": 0,
      "output_tokens": 0,
      "total_tokens": 0,
      "estimated_cost_usd": 0.0,
      "latency": {
        "calls": 0,
        "average_ms": 0,
        "p50_ms": 0,
        "p95_ms": 0
      },
      "models_used": []
    },
    "cache": {
      "cache_path": "/home/sagemaker-user/complaintiq/state/llm_cache.sqlite",
      "cached_responses": 4
    },
    "models_wi

In [30]:
checks = {
    "Configuration loaded": REGION == "us-east-1",
    "Worker model configured": bool(WORKER_MODEL),
    "Judge model configured": bool(JUDGE_MODEL),
    "Worker Pydantic test passed": isinstance(
        worker_result, LLMTestResult
    ),
    "Judge Pydantic test passed": isinstance(
        judge_result, LLMTestResult
    ),
    "Cache test passed": (
        first_response == second_response
        and after_second == after_first
    ),
    "SQLite cache exists": llm.CACHE_PATH.exists(),
    "Usage report available": bool(get_usage_report()),
    "Reusable module exists": (SRC / "llm.py").exists(),
    "Evidence report exists": report_path.exists()
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

assert all(checks.values()), "Phase 4 acceptance check failed"

print("\nPASS — PHASE 4 TECHNICAL ACCEPTANCE CHECK")

PASS | Configuration loaded
PASS | Worker model configured
PASS | Judge model configured
PASS | Worker Pydantic test passed
PASS | Judge Pydantic test passed
PASS | Cache test passed
PASS | SQLite cache exists
PASS | Usage report available
PASS | Reusable module exists
PASS | Evidence report exists

PASS — PHASE 4 TECHNICAL ACCEPTANCE CHECK
